In [0]:
%sql
WITH
offer_windows AS (
    SELECT * FROM VALUES
        ('2024 Offer A', DATE'2024-11-29', DATE'2024-12-02'),
        ('2024 Offer B', DATE'2024-12-03', DATE'2024-12-31'),
        ('2025 Offer A', DATE'2025-05-09', DATE'2025-05-29'),
        ('2025 Offer B', DATE'2025-11-28', DATE'2025-12-01'),
        ('2025 Offer C', DATE'2025-12-11', DATE'2025-12-26'),
        ('2026 Offer A', DATE'2026-05-01', DATE'2026-05-31'),
        ('2026 Offer B', DATE'2026-06-15', DATE'2026-07-14')
    AS w(window_name, start_date, end_date)
),
events AS (
    SELECT cc_user_profile_id, event_type, plan_id, changed, changed_from, changed_to,
           CAST(reporting_datetime AS DATE)               AS rpt_date,
           CAST(from_unixtime(record_occurred_at) AS DATE) AS occurred_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE cc_user_profile_id IS NOT NULL
),
data_end AS (
    SELECT MAX(rpt_date) AS last_date FROM events
),
periods AS (
    SELECT c.cal_date,
           YEAR(c.cal_date) AS yr,
           CASE WHEN w.window_name IS NOT NULL THEN 'Offer' ELSE 'Non-Offer' END AS period_type
    FROM (SELECT explode(sequence(DATE'2024-01-01', last_date, INTERVAL 1 DAY)) AS cal_date FROM data_end) c
    LEFT JOIN offer_windows w ON c.cal_date BETWEEN w.start_date AND w.end_date
),

-- ============ BUCKETS (per year + period) ============
first_created AS (
    SELECT cc_user_profile_id, MIN(rpt_date) AS first_created_date
    FROM events
    WHERE event_type = 'subscription_created'
    GROUP BY cc_user_profile_id
),
p_took_effect AS (
    SELECT p.yr, p.period_type, e.cc_user_profile_id, MIN(e.rpt_date) AS event_date
    FROM events e
    JOIN periods p ON e.rpt_date = p.cal_date
    WHERE e.event_type = 'subscription_cancelled' AND e.plan_id = 'monthly1' AND e.changed = TRUE
    GROUP BY p.yr, p.period_type, e.cc_user_profile_id
),
p_initiated AS (
    SELECT p.yr, p.period_type, e.cc_user_profile_id, MIN(e.occurred_date) AS event_date
    FROM events e
    JOIN periods p ON e.occurred_date = p.cal_date
    WHERE e.event_type = 'subscription_changed' AND e.changed = TRUE
      AND e.changed_from != e.changed_to AND e.changed_to = 'annual1'
    GROUP BY p.yr, p.period_type, e.cc_user_profile_id
),
p_signups AS (
    SELECT s.yr, s.period_type, s.cc_user_profile_id, s.event_date,
           CASE WHEN s.event_date = f.first_created_date THEN 'net_new' ELSE 'returning' END AS bucket
    FROM (
        SELECT p.yr, p.period_type, e.cc_user_profile_id, MIN(e.rpt_date) AS event_date
        FROM events e
        JOIN periods p ON e.rpt_date = p.cal_date
        WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
        GROUP BY p.yr, p.period_type, e.cc_user_profile_id
    ) s
    LEFT JOIN first_created f ON f.cc_user_profile_id = s.cc_user_profile_id
    LEFT ANTI JOIN p_took_effect u
      ON u.yr = s.yr AND u.period_type = s.period_type AND u.cc_user_profile_id = s.cc_user_profile_id
),
annual_starts AS (
    SELECT cc_user_profile_id, rpt_date AS start_date FROM events
    WHERE event_type = 'subscription_created' AND plan_id = 'annual1' AND rpt_date IS NOT NULL
    UNION
    SELECT cc_user_profile_id, rpt_date FROM events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE AND rpt_date IS NOT NULL
),
annual_payments AS (
    SELECT DISTINCT cc_user_profile_id, rpt_date FROM events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1' AND rpt_date IS NOT NULL
),
renewal_payments AS (
    SELECT ap.cc_user_profile_id, ap.rpt_date
    FROM annual_payments ap
    JOIN annual_starts s ON s.cc_user_profile_id = ap.cc_user_profile_id AND s.start_date <= ap.rpt_date
    GROUP BY ap.cc_user_profile_id, ap.rpt_date
    HAVING DATEDIFF(ap.rpt_date, MAX(s.start_date)) >= 360
),
p_renewals AS (
    SELECT p.yr, p.period_type, r.cc_user_profile_id, MIN(r.rpt_date) AS event_date
    FROM renewal_payments r
    JOIN periods p ON r.rpt_date = p.cal_date
    GROUP BY p.yr, p.period_type, r.cc_user_profile_id
),
all_members AS (
    SELECT yr, period_type, cc_user_profile_id, event_date, bucket,                 1 AS bucket_rank FROM p_signups
    UNION ALL
    SELECT yr, period_type, cc_user_profile_id, event_date, 'upgrade_initiated',    2 FROM p_initiated
    UNION ALL
    SELECT yr, period_type, cc_user_profile_id, event_date, 'upgrade_took_effect',  3 FROM p_took_effect
    UNION ALL
    SELECT yr, period_type, cc_user_profile_id, event_date, 'renewal_in_period',    4 FROM p_renewals
),
-- One bucket per member per year+period: earliest action wins
people_dedup AS (
    SELECT *
    FROM (
        SELECT a.*,
               ROW_NUMBER() OVER (PARTITION BY a.yr, a.period_type, a.cc_user_profile_id
                                  ORDER BY a.event_date, a.bucket_rank) AS rn
        FROM all_members a
    )
    WHERE rn = 1
),

-- ============ NEXT-YEAR RENEWAL ============
initiated_took_effect AS (
    SELECT d.yr, d.period_type, d.cc_user_profile_id, MIN(e.rpt_date) AS took_effect_date
    FROM people_dedup d
    JOIN events e
      ON e.cc_user_profile_id = d.cc_user_profile_id
     AND e.event_type = 'subscription_cancelled' AND e.plan_id = 'monthly1' AND e.changed = TRUE
     AND e.rpt_date BETWEEN d.event_date AND DATE_ADD(d.event_date, 35)
    WHERE d.bucket = 'upgrade_initiated'
    GROUP BY d.yr, d.period_type, d.cc_user_profile_id
),
anchors AS (
    SELECT d.yr, d.period_type, d.bucket, d.bucket_rank, d.cc_user_profile_id,
           CASE WHEN d.bucket = 'upgrade_initiated' THEN i.took_effect_date
                ELSE d.event_date END AS annual_start      -- renewals: anchor = renewal payment date
    FROM people_dedup d
    LEFT JOIN initiated_took_effect i
      ON i.yr = d.yr AND i.period_type = d.period_type AND i.cc_user_profile_id = d.cc_user_profile_id
    WHERE d.period_type = 'Non-Offer' AND d.yr IN (2024, 2025)
),
next_year_renewals AS (
    SELECT DISTINCT a.yr, a.period_type, a.cc_user_profile_id
    FROM anchors a
    JOIN annual_payments ap
      ON ap.cc_user_profile_id = a.cc_user_profile_id
     AND DATEDIFF(ap.rpt_date, a.annual_start) BETWEEN 360 AND 385
)
SELECT a.yr AS cohort_year,
       COALESCE(a.bucket, 'ALL') AS bucket,
       COUNT(DISTINCT a.cc_user_profile_id)                                                 AS members,
       COUNT(DISTINCT CASE WHEN a.annual_start IS NULL THEN a.cc_user_profile_id END)       AS upgrade_not_completed,
       COUNT(DISTINCT CASE WHEN DATE_ADD(a.annual_start, 385) >  de.last_date
                           THEN a.cc_user_profile_id END)                                   AS not_yet_eligible,
       COUNT(DISTINCT CASE WHEN DATE_ADD(a.annual_start, 385) <= de.last_date
                           THEN a.cc_user_profile_id END)                                   AS eligible_for_renewal,
       COUNT(DISTINCT CASE WHEN DATE_ADD(a.annual_start, 385) <= de.last_date
                            AND r.cc_user_profile_id IS NOT NULL
                           THEN a.cc_user_profile_id END)                                   AS renewed_next_year,
       ROUND(
         COUNT(DISTINCT CASE WHEN DATE_ADD(a.annual_start, 385) <= de.last_date
                              AND r.cc_user_profile_id IS NOT NULL
                             THEN a.cc_user_profile_id END)
         / NULLIF(COUNT(DISTINCT CASE WHEN DATE_ADD(a.annual_start, 385) <= de.last_date
                                      THEN a.cc_user_profile_id END), 0), 3)                AS renewal_rate
FROM anchors a
CROSS JOIN data_end de
LEFT JOIN next_year_renewals r
  ON r.yr = a.yr AND r.period_type = a.period_type AND r.cc_user_profile_id = a.cc_user_profile_id
GROUP BY GROUPING SETS ((a.yr, a.bucket, a.bucket_rank), (a.yr))
ORDER BY cohort_year, COALESCE(a.bucket_rank, 99), bucket